In [ ]:
import torch 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from cdft.dft3d_pcsaft import dft_core
from cdft.pcsaft_eos import pcsaft

torch.set_default_dtype(torch.float64)
device = torch.device('cuda')

In [ ]:
m = torch.tensor([1.0, 1.6069])
sigma = torch.tensor([3.7039, 3.5206])
epsilon = torch.tensor([150.03, 191.42])
parameters = {'m':m, 'sigma':sigma, 'epsilon':epsilon}

In [ ]:
structure = Structure.from_file('structures/IRMOF-1.cif')
print('formula:        ', structure.formula)
print('num_sites:      ', structure.num_sites)
print('lattice_lengths:', structure.lattice.lengths)

In [ ]:
T = 300.0
system_size = np.array([l for l in structure.lattice.lengths])
points = np.array([64, 64, 64])
dft = dft_core(parameters, T, system_size, points, device)

In [ ]:
forcefield = pd.DataFrame()
forcefield['type'] = ['Zn','H','C','O']
forcefield['sigma'] = np.array([4.045, 2.846, 3.47299, 3.033])
forcefield['epsilon'] = np.array([27.677, 7.6489, 47.8562, 48.1581])
forcefield['mass'] = np.array([65.38, 1.00784, 12.0107, 15.999])

def lj_potential(r,sigma,epsilon):
    return 4.0*epsilon*((sigma/r)**12-(sigma/r)**6) 

Vext = torch.zeros((dft.Nc, points[0], points[1], points[2]),device=device)
U = torch.zeros_like(dft.X)
rc = 12.0
for i in range(dft.Nc):
    for k, site in enumerate(structure):
        sigmas = float(forcefield['sigma'][forcefield['type']==site.species_string].values[0])
        epsilons = float(forcefield['epsilon'][forcefield['type']==site.species_string].values[0])
        sigmasf = 0.5*(sigma[i].numpy()+sigmas) 
        epsilonsf = np.sqrt(epsilon[i].numpy()*epsilons)
        rx = dft.X-structure.cart_coords[k,0] 
        ry = dft.Y-structure.cart_coords[k,1] 
        rz = dft.Z-structure.cart_coords[k,2] 
        rx -= system_size[0]*(rx/system_size[0]).round()
        ry -= system_size[1]*(ry/system_size[1]).round()
        rz -= system_size[2]*(rz/system_size[2]).round()
        r = torch.sqrt(rx**2+ry**2+rz**2)
        U = m[i]*lj_potential(r,sigmasf,epsilonsf)
        U[r==0] = np.inf
        U[r>rc] = 0.0
        Vext[i] += U

In [ ]:
plt.rcParams.update({'text.usetex':True, 
'font.family':'serif', 
'font.size':18, 
'axes.linewidth':1.1, 
'lines.linewidth':1.6,
'legend.fontsize': 18,
'legend.frameon':False
#'figure.figsize':(7.9, 6.1)
})


plt.figure()
c=plt.pcolormesh(dft.X[:,:,points[0]//2].cpu(),dft.Y[:,:,points[1]//2].cpu(),Vext[1,:,:,points[2]//2].cpu()/T, vmax=50.0, cmap='jet')
plt.colorbar(label=r'$V_{\mathrm{ext}}/k_B T$')
plt.xlabel(r'$x$ (\AA{})')
plt.ylabel(r'$y$ (\AA{})')

In [ ]:
P = torch.hstack((torch.arange(1e5,1e6,1e5), torch.range(1e6,1e7,1e6)))

bulk_density = torch.empty_like(P)
composition = torch.tensor([0.4,0.6])

eos = pcsaft(parameters, T)
bulk_density[0] = eos.density(P[0],composition,'vap')
for i in range(1,len(P)):
    bulk_density[i] = eos.density(P[i],composition,bulk_density[i-1])

In [ ]:
dft.initial_condition(bulk_density[0],composition,Vext)

In [ ]:
Nads = torch.empty((dft.Nc,len(P)))
for i in range(len(P)):
    dft.equilibrium_density_profile(bulk_density[i],composition,fmt='ASWB',solver='anderson',
                                    anderson_mmax=10,anderson_damping=0.2,tol=1e-6,logoutput=False)
    for j in range(dft.Nc):
        Nads[j,i] = dft.total_molecules[j]
    print(dft.it,dft.error.numpy(),1e-5*P[i].numpy(),Nads[:,i].numpy())
    if np.isnan(dft.error.numpy()): break

In [ ]:
data = pd.read_pickle('data/isotherm_methane_ethane.pkl')

plt.plot(P*1e-5, Nads[0], 'C0-', linewidth=1.8, label='methane')
plt.plot(P*1e-5, Nads[1], 'k-', linewidth=1.8, label='ethane')
plt.plot(data['pressure'], data['methane'], 'C0o', markersize=7, markeredgewidth=1.6, mfc = 'none')
plt.plot(data['pressure'], data['ethane'], 'ko', markersize=7, markeredgewidth=1.6, mfc = 'none')
plt.xlabel(r'$P$ (bar)')
plt.ylabel(r'$N$ (molecules/u.c.)')
plt.legend()